# FIRELINE — Kabupaten/Kota Drill-Down

Working notebook for Phase 1 onward. This notebook is the single working surface for validating the new administrative boundaries, normalizing population context, assigning grid-days, producing kabupaten summaries, and preparing dashboard outputs.

The existing grid-day priority score is kept unchanged.

## Working rules

- Primary boundaries come from `data/additions/*.geojson`.
- `kode_kk` is the candidate canonical kabupaten ID.
- Grid-day assignment uses the representative observed point.
- Population is contextual and never enters the priority score.
- Validation results and derived tables should be saved under `review_outputs/`.
- Reusable implementation should be promoted into the existing `fireline/` package only after the notebook logic is validated.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json

import geopandas as gpd
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ADDITIONS = ROOT / 'data' / 'additions'
OUTPUT = ROOT / 'review_outputs'
OUTPUT.mkdir(exist_ok=True)
ADDITIONS

## Phase 1 — Boundary inventory

In [ ]:
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

boundary_rows = []
boundary_frames = []
for path in sorted(ADDITIONS.glob('*.geojson')):
    frame = gpd.read_file(path).to_crs(4326)
    boundary_frames.append(frame.assign(_source_file=path.name))
    boundary_rows.append({
        'file': path.name,
        'features': len(frame),
        'crs': str(frame.crs),
        'geometry_types': sorted(frame.geometry.geom_type.dropna().unique()),
        'invalid_geometries': int((~frame.geometry.is_valid).sum()),
        'empty_geometries': int(frame.geometry.is_empty.sum()),
        'columns': [c for c in frame.columns if c != 'geometry'],
        'sha256': sha256(path),
    })
boundary_inventory = pd.DataFrame(boundary_rows)
boundary_inventory

In [ ]:
kabupaten = gpd.GeoDataFrame(pd.concat(boundary_frames, ignore_index=True), crs=4326)
required = {'kode_kk', 'kode_prov', 'kab_kota', 'provinsi', 'geometry'}
missing = required - set(kabupaten.columns)
assert not missing, f'Missing boundary fields: {sorted(missing)}'
assert kabupaten.kode_kk.notna().all()
assert kabupaten.kode_kk.astype(str).is_unique
assert kabupaten.geometry.is_valid.all()
kabupaten = kabupaten.rename(columns={
    'kode_kk': 'kabupaten_code',
    'kode_prov': 'province_code',
    'kab_kota': 'kabupaten_name',
    'provinsi': 'province_name',
})
kabupaten[['province_code', 'province_name', 'kabupaten_code', 'kabupaten_name', '_source_file']].head()

## Phase 1 — Population inventory

Population sources have different layouts and units. The next cells should normalize only confirmed kabupaten-level sources. Province totals and unmatched rows remain audit outputs, not silent joins.

In [ ]:
population_files = sorted(ADDITIONS.glob('*.csv')) + sorted(ADDITIONS.glob('*.xlsx'))
[(p.name, p.suffix, p.stat().st_size, sha256(p)) for p in population_files]

## Later phases

1. Normalize administrative IDs and names.
2. Validate point-to-kabupaten attribution, including border cases.
3. Load existing grid-day outputs without changing score calculations.
4. Produce `kabupaten_day_summary`.
5. Prepare the dashboard payload and hierarchical UI behavior.
6. Run regression checks against the current province-level outputs.